# Homework 1

#### EE-556 Mathematics of Data - Fall 2026


In this homework, we consider a multiclass classification task modeled by multinomial (softmax) logistic regression. Your goal will be to analyze the estimator and its properties (convexity, existence/uniqueness), and to derive gradients/Hessians and smoothness bounds. The first part consists of theoretical questions only.


<div class="alert alert-info">
  ℹ️ <strong>Information on group based work:</strong>
</div>

- Only one group member needs to submit all five completed notebooks on behalf of the group.
- You may seek help outside your group, but each group must write its own answers.
- If the work was <span style="color: red;">unevenly distributed</span> among group members, let the TAs know.
- Only one member of the group is allowed to use AI. We will require sharing the conversation history with the AI in the form of a public link. If you use multiple conversations across the same or multiple tools please share all of them. Name the person in your group who is allowed to use AI. We encourage you to use the AI to help you understand the material, but we ask you to write the code and theory solutions by yourself. Part 5 permits AI usage as specified in its instructions.

<div style="border: 1px solid #f00; background-color: #fdd; padding: 10px; border-radius: 5px;">
  ⚠️ Do not forget: List each group member's name and SCIPER number.
</div>


Person 1 **Name**: Mohamed Taha Elallaoui || Person 1 **SCIPER**: 440640


Person 2 **Name**: Alex PETER || Person 2 **SCIPER**: 362524


Person 3 **Name**: Gabriele Tortorella || Person 3 **SCIPER**: 440657


<div style="border: 1px solid #0a0; background-color: #dfd; padding: 10px; border-radius: 5px;">
  📓 Feedback on AI use: Please use the following cell to provide feedback on the AI use in this notebook.
  
  For example, how useful were the tools to you? Which tools did you use? Did you feel like they helped you understand the material better?
</div>

Chat AI links :
-https://chatgpt.com/share/6ad5e898-9998-83ed-8636-b41e26d10ac9
-https://chat.deepseek.com/share/pwex35h1zbq9vjdgzk
-https://chat.deepseek.com/share/razvt8h31kizah0nrx

We used the tools to reexplain some methods of the course and to rewrite some of our maths latex answers.


## 1. Multiclass Softmax Logistic Regression - 30 Points


We now model multiclass classification with classes $c \in \{1,\dots,C\}$. For each sample $(\mathbf{a}_i, b_i)$ with $\mathbf{a}_i \in \mathbb{R}^p$ and $b_i \in \{1,\dots,C\}$, let $\mathbf{X} = [\mathbf{x}_1,\dots,\mathbf{x}_C] \in \mathbb{R}^{p\times C}$ be the class weight matrix. The softmax model defines

$$
\mathbb{P}(b_i = c \mid \mathbf{a}_i) = \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c)}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)}.
$$

Assume i.i.d. samples $\{(\mathbf{a}_i,b_i)\}_{i=1}^n$. Our goal is to estimate $\mathbf{X}$ by maximum likelihood (and later with an $\ell_2$ regularizer).


__(a)__ (2 points) Show that the negative log-likelihood $f$ can be written as:

$$
\begin{aligned}
 f(\mathbf{X})
 &= - \log \mathbb{P}(b_1,\dots,b_n\mid \mathbf{a}_1,\dots,\mathbf{a}_n)\\
 &= \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right].
\end{aligned}
$$



Answer : Since $(b_i \mid \mathbf{a}_i)$ are independent events one to another, we have :
$$
\begin{aligned}
 f(\mathbf{X})
 &= - \log \mathbb{P}(b_1,\dots,b_n\mid \mathbf{a}_1,\dots,\mathbf{a}_n)\\
 &= - \log \prod_{i=1}^n \mathbb{P}(b_i\mid \mathbf{a}_i) \\
 &= - \log \prod_{i=1}^n \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_{b_i})}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)} \\
 &= - \log \prod_{i=1}^n \exp(\mathbf{a}_i^\top \mathbf{x}_{b_i}) + \log \prod_{i=1}^n \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \\
 &= - \sum_{i=1}^n \log\big(\exp(\mathbf{a}_i^\top \mathbf{x}_{b_i})\big) + \sum_{i=1}^n \log\Big(\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)\Big) \\
 &= \boxed{\sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right]}
\end{aligned}
$$


__(b)__ (3 points) Show that $\mathbf{u} \mapsto \log\!\left(\sum_{k=1}^C e^{u_k}\right)$ is convex on $\mathbb{R}^C$. Then, show that $f(\mathbf{X})$ is convex.


Hint: use Jensen's inequality.

Answer : Let $\mathbf{x},\mathbf{y}\in\mathbb{R}^C$ and $\alpha\in(0,1)$ (the cases $\alpha\in\{0,1\}$ are trivial). We want to show
$$
\log\Big(\sum_{k=1}^C e^{\alpha x_k+(1-\alpha)y_k}\Big) \le \alpha\log\Big(\sum_{k=1}^C e^{x_k}\Big)+(1-\alpha)\log\Big(\sum_{k=1}^C e^{y_k}\Big) = \log\Big(\big(\sum_{k=1}^C e^{x_k}\big)^{\alpha}\big(\sum_{k=1}^C e^{y_k}\big)^{1-\alpha}\Big).
$$
Since $\log$ is increasing, $\log(p)\le\log(q) \iff p\le q$ for $p,q>0$, so it suffices to show
$$
\sum_{k=1}^C e^{\alpha x_k}e^{(1-\alpha)y_k} \le \Big(\sum_{k=1}^C e^{x_k}\Big)^{\alpha}\Big(\sum_{k=1}^C e^{y_k}\Big)^{1-\alpha}.
$$
Let $A_k=e^{x_k}>0$, $B_k=e^{y_k}>0$, $S_A=\sum_{k=1}^C A_k$, $z_k=\frac{B_k}{A_k}>0$ and $w_k=\frac{A_k}{S_A}$, so that $w_k>0$ and $\sum_{k=1}^C w_k=1$. Then
$$
\begin{aligned}
\sum_{k=1}^C A_k^{\alpha}B_k^{1-\alpha} &= \sum_{k=1}^C A_k\,A_k^{\alpha-1}B_k^{1-\alpha} = \sum_{k=1}^C A_k\Big(\frac{B_k}{A_k}\Big)^{1-\alpha} = \sum_{k=1}^C A_k z_k^{1-\alpha} \\
&= S_A\sum_{k=1}^C w_k z_k^{1-\alpha}.
\end{aligned}
$$
For $0<\alpha<1$, the function $z\mapsto z^{1-\alpha}$ is concave on $(0,\infty)$ (equivalently $-z^{1-\alpha}$ is convex). Applying Jensen's inequality with weights $w_k$:
$$
\sum_{k=1}^C w_k z_k^{1-\alpha} \le \Big(\sum_{k=1}^C w_k z_k\Big)^{1-\alpha}.
$$
Since $\sum_{k=1}^C w_k z_k = \sum_{k=1}^C \frac{A_k}{S_A}\frac{B_k}{A_k} = \frac{1}{S_A}\sum_{k=1}^C B_k$, we get
$$
\begin{aligned}
\sum_{k=1}^C A_k^{\alpha}B_k^{1-\alpha} &\le S_A\Big(\sum_{k=1}^C B_k\Big)^{1-\alpha}\frac{1}{S_A^{1-\alpha}} = S_A^{\alpha}\Big(\sum_{k=1}^C B_k\Big)^{1-\alpha} \\
&= \Big(\sum_{k=1}^C A_k\Big)^{\alpha}\Big(\sum_{k=1}^C B_k\Big)^{1-\alpha}.
\end{aligned}
$$
$$\boxed{\mathbf{u}\mapsto\log\Big(\sum_{k=1}^C e^{u_k}\Big) \text{ is convex on } \mathbb{R}^C.}$$

Now we show that $f$ is convex:
$$
f(\mathbf{X}) = \sum_{i=1}^n\Big[\log\Big(\sum_{k=1}^C e^{\mathbf{a}_i^\top\mathbf{x}_k}\Big) - \mathbf{a}_i^\top\mathbf{x}_{b_i}\Big].
$$
The map $\mathbf{X}\mapsto(\mathbf{a}_i^\top\mathbf{x}_1,\dots,\mathbf{a}_i^\top\mathbf{x}_C)$ is linear, and a convex function composed with a linear map is convex, so by the result above $\log\big(\sum_{k=1}^C e^{\mathbf{a}_i^\top\mathbf{x}_k}\big)$ is convex in $\mathbf{X}$. The term $-\mathbf{a}_i^\top\mathbf{x}_{b_i}$ is linear, hence convex. The sum of two convex functions is convex, so each term of the sum is convex, i.e. $f_i(\alpha\mathbf{X}_1+(1-\alpha)\mathbf{X}_2)\le\alpha f_i(\mathbf{X}_1)+(1-\alpha)f_i(\mathbf{X}_2)$, and a sum of $n$ convex functions is convex too.
$$\boxed{f(\mathbf{X}) \text{ is convex.}}$$


Since the negative log-likelihood $f$ is convex, every local minimizer is a global minimizer. The maximum likelihood estimation problem is
$$
\mathbf{X}^\star_{ML} = \arg\min_{\mathbf{X} \in \mathbb{R}^{p\times C}} f(\mathbf{X})
$$

But does $f$ always attain its infimum? The following three questions examine this issue.


__(c)__ (1 point) Explain the difference between infima and minima. Give an example of a convex function on $\mathbb{R}$ that does not attain its infimum.


Answer : Let $f: \mathcal{X} \to \mathbb{R}$.

- The infimum $\inf_{\mathcal{X}} f$ is the greatest lower bound of the set of values $\{f(x) : x \in \mathcal{X}\}$. It always exists in $\mathbb{R}\cup\{-\infty\}$, but it does not have to be a value taken by $f$.
- A minimum exists if the infimum is attained: there is $x^\star \in \mathcal{X}$ such that $f(x^\star) = \inf_{\mathcal{X}} f$. Then $x^\star$ is a minimiser and $\min_{\mathcal{X}} f = \inf_{\mathcal{X}} f$.

Example: $f(x) = e^{-x}$ is convex on $\mathbb{R}$ since $f''(x) = e^{-x} > 0$. We have $f(x) > 0$ for all $x$ and $f(x) \xrightarrow[x \to +\infty]{} 0$, so $\inf_{\mathbb{R}} f = 0$. However, there is no $x \in \mathbb{R}$ with $e^{-x} = 0$, so the infimum is not attained.
$$\boxed{f(x)=e^{-x} \text{ is convex on } \mathbb{R}, \ \inf_{\mathbb{R}} f = 0, \text{ but } f \text{ has no minimum.}}$$


__(d)__ (2 points) Assume there exists $\mathbf{X}_0 \in \mathbb{R}^{p\times C}$ such that for all $i$,
$$
\mathbf{a}_i^\top \mathbf{x}_{0, b_i} - \max_{k \neq b_i} \mathbf{a}_i^\top \mathbf{x}_{0,k} > 0.
$$
This is called one-versus-all complete separation in multiclass settings. Give a geometric interpretation (e.g., for $p=2$) and explain why the name is appropriate.


Answer : The condition says that there exists $\mathbf{X}_0$ such that, for every sample $i$, the score of the true class $\mathbf{a}_i^\top\mathbf{x}_{0,b_i}$ is strictly larger than the score $\mathbf{a}_i^\top\mathbf{x}_{0,k}$ of every other class $k\neq b_i$: for every point, the best score is always given by the correct class.

Geometrically, $\mathbf{X}_0$ splits the feature space into the regions
$$
R_k = \{\mathbf{a} : \mathbf{a}^\top\mathbf{x}_{0,k} > \mathbf{a}^\top\mathbf{x}_{0,l} \ \ \forall\, l\neq k\}, \qquad k=1,\dots,C,
$$
where class $k$ wins. Each $R_k$ is an intersection of half-spaces $\{\mathbf{a}:\mathbf{a}^\top(\mathbf{x}_{0,k}-\mathbf{x}_{0,l})>0\}$ whose boundaries pass through the origin (there is no bias term), so $R_k$ is a convex cone with tip at the origin. Indeed, if the condition holds for $\mathbf{a}_i$, it also holds for $t\,\mathbf{a}_i$ with any $t>0$:
$$
(t\,\mathbf{a}_i)^\top\mathbf{x}_{0,b_i} - \max_{k\neq b_i}(t\,\mathbf{a}_i)^\top\mathbf{x}_{0,k} = t\Big(\mathbf{a}_i^\top\mathbf{x}_{0,b_i} - \max_{k\neq b_i}\mathbf{a}_i^\top\mathbf{x}_{0,k}\Big) > 0,
$$
so only the direction of a point matters.

For $p=2$, the samples are points in the plane and the whole plane is divided into angular sectors starting from $(0,0)$, each of angle at most $180^\circ$, separated by rays from the origin along which two class scores are equal. Complete separation means that every sample point lies strictly inside the sector of its own class. In particular, since each $R_k$ is a single convex sector, all the points of one class must lie in the same sector: a class split into two disjoint angular regions cannot be separated in this way.

The name is appropriate because:
- complete: the condition holds for every sample, so every training point is classified correctly, with a strict margin;
- one versus all: the true class beats all the other classes at once (the $\max_{k\neq b_i}$), so each class is separated from all the others simultaneously by the boundaries of its sector.


Complete separation allows perfect classification of the training data. However, as you will show next, it prevents the existence of a finite maximum likelihood estimate.


__(e)__ (3 points) In a one-versus-all complete separation setting (as in (d)), prove that $f$ does not attain its minimum. Hint: consider $f(\alpha \mathbf{X}_0)$ as $\alpha \to +\infty$ and compare it to $f(\mathbf{X}_0)$.


Answer : For each sample $i$, let $s_{k,i} = \mathbf{a}_i^\top\mathbf{x}_k$ be the score of class $k$, and $s_{b_i,i} = \mathbf{a}_i^\top\mathbf{x}_{b_i}$ the score of the true class. Then
$$
\begin{aligned}
f(\mathbf{X}) &= \sum_{i=1}^n\Big[-s_{b_i,i} + \log\big(e^{s_{1,i}}+e^{s_{2,i}}+\dots+e^{s_{C,i}}\big)\Big] \\
&= \sum_{i=1}^n\Big[\log e^{-s_{b_i,i}} + \log\Big(\sum_{k=1}^C e^{s_{k,i}}\Big)\Big]
= \sum_{i=1}^n \log\Big(e^{-s_{b_i,i}}\sum_{k=1}^C e^{s_{k,i}}\Big).
\end{aligned}
$$
One term of the sum over $k$ is the true class $k=b_i$, so we can extract it and rewrite:
$$
f(\mathbf{X}) = \sum_{i=1}^n \log\Big(e^{-s_{b_i,i}}e^{s_{b_i,i}} + \sum_{k\neq b_i} e^{s_{k,i}-s_{b_i,i}}\Big)
= \sum_{i=1}^n \log\Big(1 + \sum_{k\neq b_i} e^{-d_{k,i}}\Big),
$$
where $d_{k,i} = s_{b_i,i} - s_{k,i}$ measures how much the true class $b_i$ beats class $k$.

Since $e^{-d_{k,i}} > 0$ for all $d_{k,i}$, each logarithm is $> \log 1 = 0$, so
$$
f(\mathbf{X}) > 0 \quad \forall\, \mathbf{X}\in\mathbb{R}^{p\times C}.
$$

For $\mathbf{X}_0$ as in (d), the separation condition gives $d_{k,i} = \mathbf{a}_i^\top\mathbf{x}_{0,b_i} - \mathbf{a}_i^\top\mathbf{x}_{0,k} > 0$ for all $i$ and all $k\neq b_i$. If we take $\alpha\mathbf{X}_0$ with $\alpha>0$, all scores are multiplied by $\alpha$, hence $d_{k,i}$ becomes $\alpha\, d_{k,i}$ and
$$
f(\alpha\mathbf{X}_0) = \sum_{i=1}^n \log\Big(1 + \sum_{k\neq b_i} e^{-\alpha d_{k,i}}\Big).
$$
Each term $e^{-\alpha d_{k,i}}$ decreases strictly with $\alpha$ (so $f(\alpha\mathbf{X}_0) < f(\mathbf{X}_0)$ for $\alpha>1$), and since $d_{k,i}>0$,
$$
e^{-\alpha d_{k,i}} \xrightarrow[\alpha\to\infty]{} 0 \quad\Longrightarrow\quad f(\alpha\mathbf{X}_0) \xrightarrow[\alpha\to\infty]{} \log 1 = 0.
$$
So $f$ gets arbitrarily close to $0$, and since $f>0$ everywhere, its infimum is $0$. But $f$ never equals $0$, so no $\mathbf{X}$ attains the infimum.
$$\boxed{\inf_{\mathbf{X}} f(\mathbf{X}) = 0 \text{ is not attained: } f \text{ has no minimum under complete separation.}}$$


We resolve this issue by adding a regularizer. Consider the regularized function

$$
 f_\mu(\mathbf{X}) = f(\mathbf{X}) + \frac{\mu}{2} \|\mathbf{X}\|_F^2, \quad \mu > 0.
$$

__(f)__ (2 points) Show that the gradient with respect to $\mathbf{X}$ of $f_\mu$ can be expressed as
$$
 \nabla_{\mathbf{X}} f_\mu(\mathbf{X}) = \sum_{i=1}^n \mathbf{a}_i \big( \mathbf{p}_i - \mathbf{e}_{b_i} \big)^\top + \mu \mathbf{X},
$$
where $\mathbf{e}_{b_i} \in \mathbb{R}^C$ is the [one-hot vector](https://en.wikipedia.org/wiki/One-hot) for class $b_i$, $\mathbf{p}_i \in \mathbb{R}^C$ has entries $p_{i,c} = \mathbb{P}(b_i=c\mid \mathbf{a}_i)$ under the softmax model, and $\mathbf{a}_i(\mathbf{p}_i - \mathbf{e}_{b_i})^\top\in\mathbb{R}^{p\times C}$ denotes the outer product.



Answer :
$$
 \nabla_{\mathbf{X}} f_\mu(\mathbf{X}) = \nabla_{\mathbf{X}}f(\mathbf{X}) + \nabla_{\mathbf{X}}\frac{\mu}{2} \|\mathbf{X}\|_F^2 = \nabla_{\mathbf{X}}f(\mathbf{X}) + \mu \mathbf{X}, \\
 \nabla_{\mathbf{X}} f(\mathbf{X}) = \nabla_{\mathbf{X}} \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right] \\
 \nabla_{\mathbf{X}} f(\mathbf{X}) = \sum_{i=1}^n \left[ -\mathbf{a}_i \mathbf{e}_{b_i}^\top + \nabla_{\mathbf{X}} \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right] \\
  \nabla_{\mathbf{X}} f(\mathbf{X}) = \sum_{i=1}^n \left[ -\mathbf{a}_i \mathbf{e}_{b_i}^\top + \mathbf{a}_i \sum_{c=1}^C \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_{c})\mathbf{e}_{c}^\top}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)} \right] \\
 \nabla_{\mathbf{X}} f(\mathbf{X}) = \sum_{i=1}^n \mathbf{a}_i \big( \mathbf{p}_i - \mathbf{e}_{b_i} \big)^\top \\
 \boxed{\nabla_{\mathbf{X}} f_\mu(\mathbf{X}) = \sum_{i=1}^n \mathbf{a}_i \big( \mathbf{p}_i - \mathbf{e}_{b_i} \big)^\top + \mu \mathbf{X}}
$$

__(g)__ (4 points) Use row-wise vectorization of $\mathbf{X}$, obtained by stacking its rows one after another into a vector in $\mathbb{R}^{pC}$. Show that the Hessian of $f_\mu$ in these coordinates can be written as
<a id="eq-softmax-hessian"></a>

$$
 \nabla^2 f_\mu(\mathbf{X}) = \sum_{i=1}^n (\mathbf{a}_i\mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big) + \mu \mathbf{I}_{pC},\tag{1}
$$
where $\otimes$ is the Kronecker product, and $\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top$ is the softmax Jacobian, which is positive semidefinite.



Answer : Let's isolate the problem in row vectorization, and derive also by row vectorization of X :
$$
 \nabla^2 f_\mu(\mathbf{X}) = \frac{\partial}{\partial \text{vec}_r(\mathbf{X})} \text{vec}_r\!\left(\nabla_{\mathbf{X}} f_\mu(\mathbf{X})\right) \\
 \frac{\partial}{\partial \text{vec}_r(\mathbf{X})}  \text{vec}_r(\sum_{i=1}^n \mathbf{a}_i \big( \mathbf{p}_i - \mathbf{e}_{b_i} \big)^\top) = \sum_{i=1}^n (\mathbf{a}_i \otimes I_C) \frac{\partial}{\partial \text{vec}_r(\mathbf{X})}\mathbf{p}_i
$$
For one row only :
$$
 (j=l) \frac{\partial p_{i,l}}{\partial \mathbf{x}_j} = p_{i,l}(1 - p_{i,l}) \mathbf{a}_i^\top \\ \\
 (j \neq l)\frac{\partial p_{i,l}}{\partial \mathbf{x}_j} = (-p_{i,l} p_{i,j}) \mathbf{a}_i^\top
$$
Thus, for the full row derivation:
$$
\frac{\partial p_{i,l}}{\partial \text{vec}_r(\mathbf{X})} = \begin{bmatrix} p_{i,l}(\delta_{l,1}-p_{i,1})a_{i,1} & \dots & p_{i,l}(\delta_{l,C}-p_{i,C})a_{i,1} & \dots & p_{i,l}(\delta_{l,1}-p_{i,1})a_{i,p} & \dots & p_{i,l}(\delta_{l,C}-p_{i,C})a_{i,p} \end{bmatrix}
$$
By stacking all rows : 
$$
\frac{\partial \mathbf{p}_i}{\partial \text{vec}_r(\mathbf{X})} = \mathbf{a}_i^\top \otimes \begin{bmatrix} p_{i,1}(1-p_{i,1}) & -p_{i,1}p_{i,2} & \dots & -p_{i,1}p_{i,C} \\ -p_{i,2}p_{i,1} & p_{i,2}(1-p_{i,2}) & \dots & -p_{i,2}p_{i,C} \\ \vdots & \vdots & \ddots & \vdots \\ -p_{i,C}p_{i,1} & -p_{i,C}p_{i,2} & \dots & p_{i,C}(1-p_{i,C}) \end{bmatrix}
$$
And so we finally get : 
$$
 \nabla^2 f_\mu(\mathbf{X}) = \sum_{i=1}^n (\mathbf{a}_i \otimes I_C) \big(\mathbf{a}_i^\top \otimes ( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top ) \big) + \mu \mathbf{I}_{pC} \\
\boxed{\nabla^2 f_\mu(\mathbf{X}) = \sum_{i=1}^n (\mathbf{a}_i\mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big) + \mu \mathbf{I}_{pC},\tag{1}}
$$

__(h)__ (2 points) Show that $f_\mu$ is $\mu$-strongly convex.


Answer : To show that the softmax Jacobian is positive semidefinite, take any $\mathbf{z}\in\mathbb{R}^C$ and set $m=\sum_{c=1}^C p_{i,c}z_c$. Since $\sum_{c=1}^C p_{i,c}=1$,
$$
\begin{aligned}
\mathbf{z}^\top\big(\operatorname{Diag}(\mathbf{p}_i)-\mathbf{p}_i\mathbf{p}_i^\top\big)\mathbf{z}
&= \sum_{c=1}^C p_{i,c}z_c^2 - \left(\sum_{c=1}^C p_{i,c}z_c\right)^2 \\
&= \sum_{c=1}^C p_{i,c}z_c^2 - 2m\sum_{c=1}^C p_{i,c}z_c + m^2 \\
&= \sum_{c=1}^C p_{i,c}(z_c-m)^2 \geq 0,
\end{aligned}
$$
because each $p_{i,c}\geq 0$. Thus the softmax Jacobian is positive semidefinite. Since $\mathbf{a}_i\mathbf{a}_i^\top$ is positive semidefinite too, their Kronecker product in (1) is positive semidefinite. Hence $\nabla^2 f_\mu(\mathbf{X})\succeq\mu\mathbf{I}_{pC}$, so $f_\mu$ is $\mu$-strongly convex.



__(i)__ (2 points) Is it possible for a strongly convex function to not attain its minimum? Justify your reasoning (you may assume the domain is $\mathbb{R}^{p\times C}$).


Answer : It is not possible, since f is strongly convex we have for X and Y in $\mathbb{R}^{p\times C}$ :

$$
f_\mu(Y) \geq f_\mu(X) + \langle \nabla f_\mu(X), Y - X \rangle + \frac{\mu}{2} \|Y - X\|^2
$$
$$\fbox{We have clearly  $f_\mu(Y) \xrightarrow[|| Y || \to \infty]{} +\infty$ (In all directions!), this means the minimum is necessarily attained.}$$

We will now show that $f_\mu$ is smooth, i.e., $\nabla f_\mu$ is L-Lipschitz with respect to the Frobenius norm, with a simple conservative bound
$$
 L = \|\mathbf{A}\|_F^2 + \mu.
$$
where
$$
 \mathbf{A} = \begin{bmatrix}
  \leftarrow &  \mathbf{a}_1^\top & \rightarrow \\
  \leftarrow &  \mathbf{a}_2^\top & \rightarrow \\
   &  \ldots &  \\
  \leftarrow &  \mathbf{a}_n^\top & \rightarrow \\
 \end{bmatrix}.
$$
(You may use that the operator norm of the softmax Jacobian is bounded by $1/2$, and a looser bound $\le 1$ is acceptable for grading.)

Hint: check the properties of the spectral norm with respect to dot product, Kronecker product, and outer product.

(4 points for all three questions)


__(j-1)__ Show that $\lambda_{\max}(\mathbf{a}_i\mathbf{a}_i^T) = \left\| \mathbf{a}_i\right\|_2^2$, where $\lambda_{\max}(\cdot)$ denotes the largest eigenvalue.


Answer : 
We verify that  $\left\| \mathbf{a}_i\right\|_2^2$ is indeed an eigenvalue of  $\mathbf{a}_i\mathbf{a}_i^T$, it is.
The matrix $\mathbf{a}_i\mathbf{a}_i^T$ is symmetric, so the rank is strictly equal to the number of eigenvalues that are not equal to 0. The rank of this matrix is one because every column has $a_i$ as a factor. 

$$\fbox{So $\left\| \mathbf{a}_i\right\|_2^2$ is exactly the greatest eigenvalue.}$$




__(j-2)__ Using [(1)](#eq-softmax-hessian), show that $\lambda_{\max}(\nabla^2 f_\mu(\mathbf{X})) \leq \sum_{i=1}^{n} \|\mathbf{a}_i\|_2^2 + \mu$.

__(j-3)__ Conclude that $f_\mu$ is $L$-smooth for $L = \|\mathbf{A}\|_F^2 + \mu$.


$ \|\mathbf{A}\|_F^2 = \sum_{i=1}^{n} \|\mathbf{a}_i\|_2^2 \\
LI \succeq \nabla^2 f_\mu(\mathbf{X}) \\
\fbox{So f is indeed L-smooth}$


__(l)__ (3 points) KL divergence and NLL. Let $q_i=(q(c\mid\mathbf{a}_i))_{c=1}^C$ be the true conditional label distribution and $\mathbf{p}_i$ the model softmax distribution. Write $\mathrm{KL}(q_i\,\|\,\mathbf{p}_i)$ and show that minimizing its average over the samples is equivalent to minimizing the expected average negative log-likelihood, where the expectation is over labels drawn from $q$ with the features held fixed. Explain how $f(\mathbf{X})/n$ from (a) estimates this expected loss.

*Write your answer here.*


The unregularized maximum likelihood problem may have no finite minimizer. Adding a quadratic regularizer with $\mu>0$ gives a unique regularized estimate, defined by the smooth, strongly convex problem:
<a id="eq-regularized-estimator"></a>

$$
 \mathbf{X}^\star = \arg\min_{\mathbf{X} \in \mathbb{R}^{p\times C}} f(\mathbf{X}) + \frac{\mu}{2}\|\mathbf{X}\|_F^2.\tag{2}
$$


## Binary logistic regression (specialization for Part 2)

While this part analyzed the multiclass (softmax) setting, in the next exercise we will continue under the simplified two-class case.

Let labels be $b_i \in \{-1, +1\}$, features $\mathbf{a}_i \in \mathbb{R}^p$, and weight vector $\mathbf{x} \in \mathbb{R}^p$. Define the sigmoid
$$
\sigma(t) = \frac{1}{1+e^{-t}}.
$$
Model the conditional distribution as
$$
\mathbb{P}(b_i = j \mid \mathbf{a}_i) = \sigma\big(j\, \mathbf{a}_i^\top \mathbf{x}\big), \quad j \in \{-1,+1\}.
$$
The likelihood over i.i.d. samples $\{(\mathbf{a}_i, b_i)\}_{i=1}^n$ is
$$
\mathcal{L}(\mathbf{x}) = \prod_{i=1}^n \sigma\big(b_i\, \mathbf{a}_i^\top \mathbf{x}\big),
$$
so the negative log-likelihood is
$$
 f(\mathbf{x}) = -\log \mathcal{L}(\mathbf{x}) = \sum_{i=1}^n \log\big(1 + e^{-b_i\, \mathbf{a}_i^\top \mathbf{x}}\big).
$$

__(m)__ (2 points) Show that the gradient of the negative log-likelihood is the standard binary logistic regression gradient:
$$
\nabla f(\mathbf{x}) = \sum_{i=1}^n \big(-b_i\, \sigma(-b_i\, \mathbf{a}_i^\top \mathbf{x})\big)\, \mathbf{a}_i.
$$
(Hint: use the chain rule and $\sigma'(t) = \sigma(t)\big(1-\sigma(t)\big)$.)

We will use this binary formulation in Part 2 - First order methods.


Answer : 

$$

\nabla f(\mathbf{x}) 
= \sum_{i=1}^n \nabla\log\big(1 + e^{-b_i\, \mathbf{a}_i^\top \mathbf{x}}\big) \\
= \sum_{i=1}^n -b_i\frac{e^{-b_i\, \mathbf{a}_i^\top \mathbf{x}}}{\big(1 + e^{-b_i\, \mathbf{a}_i^\top \mathbf{x}}\big)}\mathbf{a}_i
= \sum_{i=1}^n -b_i\frac{1}{\big(1 + e^{b_i\, \mathbf{a}_i^\top \mathbf{x}}\big)}\mathbf{a}_i \\
\boxed{= \sum_{i=1}^n \big(-b_i\, \sigma(-b_i\, \mathbf{a}_i^\top \mathbf{x})\big)\, \mathbf{a}_i}
$$
